# Vortrainiertes CNN zur Erkennung von Autos

Dieses Notebook implementiert ein vortrainiertes CNN (MobileNetV2) mit Transfer Learning zur Erkennung von Autos im CIFAR-10 Datensatz.

## Einführung und theoretischer Hintergrund

Transfer Learning ist eine leistungsstarke Technik im Bereich des maschinellen Lernens, bei der ein Modell, das für eine Aufgabe trainiert wurde, als Ausgangspunkt für ein Modell für eine andere Aufgabe verwendet wird. Diese Methode ist besonders nützlich, wenn begrenzte Daten für die Zielaufgabe verfügbar sind oder wenn die Rechenressourcen für das Training eines komplexen Modells von Grund auf begrenzt sind.

Die Hauptvorteile von Transfer Learning sind:

1. **Schnellere Konvergenz**: Da das vortrainierte Modell bereits allgemeine Merkmale gelernt hat, kann das angepasste Modell schneller konvergieren.
2. **Bessere Generalisierung**: Vortrainierte Modelle haben oft eine bessere Generalisierungsfähigkeit, da sie auf großen und vielfältigen Datensätzen trainiert wurden.
3. **Weniger Trainingsdaten erforderlich**: Transfer Learning ermöglicht gute Ergebnisse auch mit kleineren Datensätzen.
4. **Geringerer Rechenaufwand**: Das Training eines angepassten Modells erfordert weniger Rechenressourcen als das Training eines Modells von Grund auf.

In diesem Notebook verwenden wir MobileNetV2, ein effizientes CNN, das auf dem ImageNet-Datensatz vortrainiert wurde. MobileNetV2 wurde von Google entwickelt und ist für mobile und eingebettete Anwendungen optimiert. Es verwendet tiefenweise trennbare Faltungen (depthwise separable convolutions), die die Anzahl der Parameter und Rechenoperationen erheblich reduzieren, während sie eine hohe Genauigkeit beibehalten.

Die Architektur von MobileNetV2 besteht aus:
- Einem vollständigen Faltungslayer am Anfang
- Mehreren Bottleneck-Blöcken mit tiefenweisen trennbaren Faltungen
- Einem Pooling-Layer und einem vollständig verbundenen Layer am Ende

Für unsere Aufgabe der Autoerkennung werden wir:
1. Das vortrainierte MobileNetV2-Modell laden
2. Die vortrainierten Schichten einfrieren, um ihre Gewichte während des Trainings beizubehalten
3. Die oberen Schichten entfernen und durch neue Schichten ersetzen, die für unsere binäre Klassifikationsaufgabe geeignet sind
4. Das angepasste Modell auf unserem CIFAR-10-Datensatz trainieren

## Überblick über die Schritte
- Laden eines vortrainierten MobileNetV2-Modells
- Anpassung des Modells für die Autoerkennung durch Transfer Learning
- Training des angepassten Modells
- Evaluierung des Modells auf Testdaten
- Visualisierung der Ergebnisse

## Importieren der benötigten Bibliotheken

Für die Implementierung des Transfer Learning mit einem vortrainierten CNN benötigen wir verschiedene Python-Bibliotheken:

- **tensorflow**: Das Framework für maschinelles Lernen, das wir für das Training und die Inferenz verwenden
- **tensorflow.keras.applications**: Enthält vortrainierte Modelle wie MobileNetV2
- **tensorflow.keras.models**: Für die Definition und Anpassung von Modellen
- **tensorflow.keras.layers**: Für die Definition neuer Schichten für unser angepasstes Modell
- **tensorflow.keras.optimizers**: Für die Optimierung während des Trainings
- **tensorflow.keras.callbacks**: Für Funktionen wie Early Stopping und Model Checkpointing
- **numpy**: Für effiziente numerische Operationen
- **matplotlib**: Für die Visualisierung der Ergebnisse
- **os**: Für Dateisystem-Operationen

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
import matplotlib.pyplot as plt

## Vorbereitung der Verzeichnisse

Bevor wir mit der Implementierung beginnen, erstellen wir Verzeichnisse für die Speicherung der Modelle und Visualisierungen. Eine gute Organisation der Projektstruktur ist wichtig für die Nachvollziehbarkeit und Wiederverwendbarkeit des Codes.

Wir erstellen zwei Verzeichnisse:
- `models_dir`: Für die Speicherung der trainierten Modelle und Checkpoints
- `visualizations_dir`: Für die Speicherung von Visualisierungen wie Lernkurven und Vorhersagen

Die Funktion `os.makedirs()` mit dem Parameter `exist_ok=True` stellt sicher, dass kein Fehler auftritt, falls die Verzeichnisse bereits existieren.

In [ ]:
# Vorbereitung der Verzeichnisse
data_dir = '../data'
models_dir = '../models'
pretrained_models_dir = os.path.join(models_dir, 'pretrained')
visualizations_dir = os.path.join(models_dir, 'visualizations')

os.makedirs(models_dir, exist_ok=True)
os.makedirs(pretrained_models_dir, exist_ok=True)
os.makedirs(visualizations_dir, exist_ok=True)

## Laden der vorbereiteten Daten

Wir laden die vorbereiteten Daten aus dem ersten Notebook. Diese Daten wurden bereits normalisiert und mit binären Labels versehen (1 für Auto, 0 für Nicht-Auto).

Da MobileNetV2 für Bilder mit einer Größe von 224x224 Pixeln trainiert wurde, während unsere CIFAR-10-Bilder nur 32x32 Pixel groß sind, müssen wir die Bilder auf die richtige Größe skalieren. Dies ist ein wichtiger Schritt, da vortrainierte Modelle oft spezifische Eingabegrößen erwarten.

In [ ]:
# Laden der vorbereiteten Daten
x_train = np.load(os.path.join(data_dir, 'x_train_normalized.npy'))
x_test = np.load(os.path.join(data_dir, 'x_test_normalized.npy'))
y_train = np.load(os.path.join(data_dir, 'y_train_binary.npy'))
y_test = np.load(os.path.join(data_dir, 'y_test_binary.npy'))

print(f"Trainingsbilder: {x_train.shape}")
print(f"Testbilder: {x_test.shape}")
print(f"Trainings-Labels: {y_train.shape}")
print(f"Test-Labels: {y_test.shape}")

## Vorbereitung der Bilder für das vortrainierte Modell

Vortrainierte Modelle wie MobileNetV2 erwarten Bilder in einer bestimmten Größe und mit einer bestimmten Vorverarbeitung. MobileNetV2 wurde auf Bildern mit einer Größe von 224x224 Pixeln trainiert, und die Pixelwerte wurden mit der `preprocess_input`-Funktion vorverarbeitet.

Wir müssen daher unsere CIFAR-10-Bilder (32x32 Pixel) auf 224x224 Pixel skalieren und die entsprechende Vorverarbeitung anwenden. Die Skalierung erfolgt mit der `tf.image.resize`-Funktion, und die Vorverarbeitung mit der `preprocess_input`-Funktion von MobileNetV2.

Die Skalierung auf eine größere Größe kann zu einem Verlust an Bildqualität führen, aber sie ist notwendig, um das vortrainierte Modell verwenden zu können. Trotz dieses potenziellen Nachteils kann Transfer Learning immer noch zu besseren Ergebnissen führen als das Training eines Modells von Grund auf, insbesondere bei begrenzten Daten.

In [ ]:
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

def prepare_images_for_mobilenet(images):
    """
    Bereitet Bilder für MobileNetV2 vor, indem sie auf 224x224 Pixel skaliert und vorverarbeitet werden.
    
    Parameter:
    - images: Bilder mit Form (N, H, W, C)
    
    Rückgabe:
    - processed_images: Vorverarbeitete Bilder mit Form (N, 224, 224, 3)
    """
    # Skalieren der Bilder auf 224x224 Pixel
    resized_images = tf.image.resize(images, (224, 224))
    
    # Vorverarbeitung für MobileNetV2
    processed_images = preprocess_input(resized_images * 255.0)  # Skalieren zurück auf [0, 255] für preprocess_input
    
    return processed_images

# Vorbereitung der Trainings- und Testbilder
x_train_mobilenet = prepare_images_for_mobilenet(x_train)
x_test_mobilenet = prepare_images_for_mobilenet(x_test)

print(f"Vorbereitete Trainingsbilder für MobileNetV2: {x_train_mobilenet.shape}")
print(f"Vorbereitete Testbilder für MobileNetV2: {x_test_mobilenet.shape}")

# Visualisierung einiger vorbereiteter Bilder
plt.figure(figsize=(10, 5))
for i in range(5):
    # Original-Bild
    plt.subplot(2, 5, i+1)
    plt.imshow(x_train[i])
    plt.title("Original")
    plt.axis('off')
    
    # Skaliertes Bild (normalisiert für die Anzeige)
    plt.subplot(2, 5, i+6)
    # Konvertieren zurück zu [0, 1] für die Anzeige
    img = (x_train_mobilenet[i] - np.min(x_train_mobilenet[i])) / (np.max(x_train_mobilenet[i]) - np.min(x_train_mobilenet[i]))
    plt.imshow(img)
    plt.title("Skaliert (224x224)")
    plt.axis('off')

plt.tight_layout()
plt.savefig(os.path.join(visualizations_dir, 'mobilenet_image_preparation.png'))
plt.show()

## Laden des vortrainierten Modells

Jetzt laden wir das vortrainierte MobileNetV2-Modell. MobileNetV2 ist ein effizientes CNN, das auf dem ImageNet-Datensatz vortrainiert wurde, der über eine Million Bilder in 1000 Kategorien enthält.

Wir laden das Modell mit den vortrainierten Gewichten (`weights='imagenet'`), aber ohne die oberen Schichten (`include_top=False`), da wir diese durch unsere eigenen Schichten für die binäre Klassifikation ersetzen werden.

Die Eingabegröße wird auf 224x224 Pixel mit 3 Farbkanälen festgelegt, was der Größe entspricht, auf die wir unsere Bilder skaliert haben.

In [ ]:
# Laden des vortrainierten MobileNetV2-Modells
base_model = MobileNetV2(
    weights='imagenet',  # Vortrainierte Gewichte auf ImageNet
    include_top=False,   # Ohne die oberen Schichten
    input_shape=(224, 224, 3)  # Eingabegröße
)

# Zusammenfassung des Basismodells anzeigen
print("Zusammenfassung des Basismodells:")
base_model.summary()

## Einfrieren der vortrainierten Schichten

Um Transfer Learning effektiv zu nutzen, frieren wir die Gewichte der vortrainierten Schichten ein, damit sie während des Trainings nicht aktualisiert werden. Dies ist wichtig, da diese Schichten bereits allgemeine Merkmale gelernt haben, die für viele Bilderkennungsaufgaben nützlich sind.

Durch das Einfrieren der vortrainierten Schichten:
1. Behalten wir das wertvolle Wissen, das das Modell auf dem ImageNet-Datensatz gelernt hat
2. Reduzieren wir die Anzahl der trainierbaren Parameter, was das Training beschleunigt
3. Verhindern wir Overfitting, insbesondere wenn unser Datensatz klein ist

In einigen Fällen kann es sinnvoll sein, einige der oberen Schichten des vortrainierten Modells für das Feintuning zu trainieren, aber für unsere Aufgabe frieren wir alle vortrainierten Schichten ein.

In [ ]:
# Einfrieren der vortrainierten Schichten
for layer in base_model.layers:
    layer.trainable = False

# Überprüfen der trainierbaren Parameter
trainable_params = sum([np.prod(layer.trainable_weights[0].shape) if layer.trainable_weights else 0 for layer in base_model.layers])
total_params = sum([np.prod(layer.weights[0].shape) if layer.weights else 0 for layer in base_model.layers])

print(f"Trainierbare Parameter im Basismodell: {trainable_params}")
print(f"Gesamtzahl der Parameter im Basismodell: {total_params}")

## Hinzufügen von benutzerdefinierten Schichten

Nachdem wir das vortrainierte Modell geladen und seine Schichten eingefroren haben, fügen wir nun unsere eigenen benutzerdefinierten Schichten hinzu, die für unsere spezifische Aufgabe der Autoerkennung trainiert werden.

Wir fügen folgende Schichten hinzu:
1. **GlobalAveragePooling2D**: Diese Schicht reduziert die räumlichen Dimensionen, indem sie den Durchschnitt über alle räumlichen Positionen für jede Feature Map berechnet. Dies reduziert die Anzahl der Parameter und hilft, Overfitting zu vermeiden.
2. **Dense**: Eine vollständig verbundene Schicht mit 128 Neuronen und ReLU-Aktivierung, die als versteckte Schicht dient.
3. **Dense**: Eine Ausgabeschicht mit einem Neuron und Sigmoid-Aktivierung für die binäre Klassifikation (Auto vs. Nicht-Auto).

Diese Architektur ermöglicht es dem Modell, die vom vortrainierten Modell extrahierten Merkmale zu nutzen und sie für unsere spezifische Klassifikationsaufgabe anzupassen.

In [ ]:
# Hinzufügen von benutzerdefinierten Schichten
x = base_model.output
x = GlobalAveragePooling2D()(x)  # Globales Average Pooling
x = Dense(128, activation='relu')(x)  # Versteckte Schicht
predictions = Dense(1, activation='sigmoid')(x)  # Ausgabeschicht für binäre Klassifikation

# Erstellen des vollständigen Modells
model = Model(inputs=base_model.input, outputs=predictions)

# Zusammenfassung des vollständigen Modells anzeigen
print("Zusammenfassung des vollständigen Modells:")
model.summary()

## Kompilieren des Modells

Bevor wir das Modell trainieren können, müssen wir es kompilieren, indem wir den Optimierer, die Verlustfunktion und die Metriken festlegen.

Für unsere binäre Klassifikationsaufgabe verwenden wir:
- **Optimierer**: Adam mit einer niedrigen Lernrate (0.0001), da wir nur die oberen Schichten trainieren und eine zu hohe Lernrate die vortrainierten Merkmale stören könnte.
- **Verlustfunktion**: Binary Cross-Entropy, die für binäre Klassifikationsprobleme geeignet ist.
- **Metriken**: Accuracy (Genauigkeit), um die Leistung des Modells während des Trainings zu überwachen.

Die Wahl dieser Hyperparameter ist wichtig für den Erfolg des Transfer Learning. Eine zu hohe Lernrate könnte zu einer Überanpassung führen, während eine zu niedrige Lernrate zu einer langsamen Konvergenz führen könnte.

In [ ]:
# Kompilieren des Modells
model.compile(
    optimizer=Adam(learning_rate=0.0001),  # Niedrige Lernrate für Transfer Learning
    loss='binary_crossentropy',  # Verlustfunktion für binäre Klassifikation
    metrics=['accuracy']  # Metrik zur Überwachung
)

## Callbacks für das Training

Callbacks sind Funktionen, die während des Trainings zu bestimmten Zeitpunkten aufgerufen werden. Sie können verwendet werden, um das Training zu überwachen, zu steuern oder zu protokollieren. Wir verwenden zwei wichtige Callbacks:

1. **Early Stopping**: Dieser Callback überwacht eine bestimmte Metrik (in unserem Fall die Validierungs-Loss) und stoppt das Training, wenn sich diese Metrik über eine bestimmte Anzahl von Epochen nicht verbessert. Dies verhindert Overfitting und spart Rechenzeit.

2. **Model Checkpoint**: Dieser Callback speichert das Modell nach jeder Epoche, wenn sich eine bestimmte Metrik verbessert hat. Wir speichern das Modell, wenn die Validierungs-Loss sinkt, und behalten so das beste Modell während des Trainings.

Diese Callbacks sind besonders nützlich für das Training von neuronalen Netzwerken, da sie helfen, die optimale Anzahl von Trainingsepochen zu finden und das beste Modell zu speichern.

In [ ]:
# Callbacks für das Training
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

checkpoint_path = os.path.join(pretrained_models_dir, 'mobilenet_best_model.h5')
model_checkpoint = ModelCheckpoint(
    filepath=checkpoint_path,
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

callbacks = [early_stopping, model_checkpoint]

## Training des Modells

Jetzt trainieren wir unser angepasstes Modell auf den vorbereiteten Daten. Während des Trainings werden nur die Gewichte der benutzerdefinierten Schichten aktualisiert, während die Gewichte der vortrainierten Schichten eingefroren bleiben.

Wichtige Parameter für das Training sind:
- **Batch Size**: Die Anzahl der Beispiele, die in einem Schritt verarbeitet werden. Ein größerer Batch führt zu einer stabileren Gradientenschätzung, benötigt aber mehr Speicher.
- **Epochs**: Die Anzahl der vollständigen Durchläufe durch den Trainingsdatensatz. Wir setzen eine hohe Zahl, aber Early Stopping wird das Training stoppen, wenn keine Verbesserung mehr auftritt.
- **Validation Split**: Der Anteil der Trainingsdaten, der für die Validierung während des Trainings verwendet wird. Dies hilft, Overfitting zu erkennen.
- **Callbacks**: Die zuvor definierten Callbacks für Early Stopping und Model Checkpointing.

Das Training kann je nach Hardware einige Zeit in Anspruch nehmen. Die Fortschrittsanzeige zeigt den Verlust und die Genauigkeit für jede Epoche sowohl für die Trainings- als auch für die Validierungsdaten.

In [ ]:
# Training des Modells
batch_size = 32
epochs = 30
validation_split = 0.2

history = model.fit(
    x_train_mobilenet, y_train,
    batch_size=batch_size,
    epochs=epochs,
    validation_split=validation_split,
    callbacks=callbacks,
    verbose=1
)

## Evaluierung des Modells

Nach dem Training evaluieren wir das Modell auf den Testdaten, um seine Generalisierungsfähigkeit zu bewerten. Die Testdaten wurden während des Trainings nicht verwendet, daher geben sie uns eine unvoreingenommene Einschätzung der Modellleistung.

Wir berechnen den Verlust und die Genauigkeit auf den Testdaten und machen Vorhersagen, die wir später für detailliertere Analysen verwenden werden. Die Vorhersagen sind Wahrscheinlichkeitswerte zwischen 0 und 1, die wir mit einem Schwellenwert (typischerweise 0.5) in binäre Klassen umwandeln können.

In [ ]:
# Evaluierung des Modells auf den Testdaten
test_loss, test_accuracy = model.evaluate(x_test_mobilenet, y_test)
print("Evaluierung auf Testdaten:")
print(f"Loss: {test_loss:.4f}")
print(f"Accuracy: {test_accuracy:.4f}\n")

# Vorhersagen für die Testdaten
y_pred_prob = model.predict(x_test_mobilenet)
y_pred = (y_pred_prob > 0.5).astype(int)
print("Vorhersagen wurden erstellt.")

## Visualisierung der Trainingsergebnisse

Die Visualisierung des Trainingsverlaufs hilft uns, das Verhalten des Modells während des Trainings zu verstehen. Wir plotten den Verlust und die Genauigkeit sowohl für die Trainings- als auch für die Validierungsdaten über die Epochen hinweg.

Diese Plots können uns wichtige Einblicke geben:
- Konvergiert das Modell? (Sinkt der Verlust kontinuierlich?)
- Gibt es Anzeichen von Overfitting? (Divergieren die Trainings- und Validierungskurven?)
- Wie viele Epochen sind optimal? (Wann beginnt der Validierungsverlust zu steigen?)

Die Visualisierungen werden auch gespeichert, um sie später in Berichten oder Präsentationen verwenden zu können.

In [ ]:
# Visualisierung des Trainingsverlaufs
plt.figure(figsize=(12, 5))

# Plot für den Verlust
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Trainingsverlauf - Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plot für die Genauigkeit
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Trainingsverlauf - Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(visualizations_dir, 'mobilenet_training_history.png'))
plt.show()

## Speichern des Modells

Obwohl wir bereits das beste Modell während des Trainings mit dem ModelCheckpoint-Callback gespeichert haben, speichern wir hier das endgültige Modell explizit. Dies ist nützlich, wenn wir später das Modell für Vorhersagen verwenden möchten, ohne es erneut trainieren zu müssen.

Das Modell wird im HDF5-Format (.h5) gespeichert, das sowohl die Architektur als auch die Gewichte des Modells enthält. Dies ermöglicht ein einfaches Laden und Verwenden des Modells in anderen Anwendungen.

In [ ]:
# Speichern des finalen Modells
final_model_path = os.path.join(pretrained_models_dir, 'mobilenet_final_model.h5')
model.save(final_model_path)
print(f"Modell wurde gespeichert unter: {final_model_path}")

## Visualisierung der Vorhersagen

Um ein besseres Verständnis für die Leistung unseres Modells zu bekommen, visualisieren wir einige Beispiele aus dem Testdatensatz zusammen mit den Vorhersagen des Modells. Wir zeigen sowohl korrekte als auch falsche Vorhersagen, um zu verstehen, wo das Modell gut funktioniert und wo es Schwierigkeiten hat.

Diese Visualisierung kann uns helfen, mögliche Muster in den Fehlern des Modells zu erkennen und Ideen für Verbesserungen zu entwickeln. Zum Beispiel könnten wir feststellen, dass das Modell Schwierigkeiten hat, Autos aus bestimmten Perspektiven oder unter bestimmten Lichtbedingungen zu erkennen.

In [ ]:
# Visualisierung einiger Vorhersagen
def visualize_predictions(x_data, y_true, y_pred, y_pred_prob, num_examples=10):
    # Zufällige Indizes auswählen
    np.random.seed(42)  # Für Reproduzierbarkeit
    indices = np.random.choice(len(y_true), size=num_examples, replace=False)
    
    # Erstellen der Visualisierung
    plt.figure(figsize=(15, 8))
    for i, idx in enumerate(indices):
        plt.subplot(2, 5, i+1)
        plt.imshow(x_data[idx])  # Originalbild anzeigen (nicht das skalierte)
        plt.axis('off')
        
        true_label = 'Auto' if y_true[idx][0] == 1 else 'Nicht-Auto'
        pred_label = 'Auto' if y_pred[idx][0] == 1 else 'Nicht-Auto'
        confidence = y_pred_prob[idx][0]
        
        color = 'green' if y_true[idx][0] == y_pred[idx][0] else 'red'
        plt.title(f"Wahr: {true_label}\nVorhersage: {pred_label}\nKonfidenz: {confidence:.2f}", color=color)
    
    plt.tight_layout()
    plt.savefig(os.path.join(visualizations_dir, 'mobilenet_predictions.png'))
    plt.show()

# Visualisierung von Vorhersagen
visualize_predictions(x_test, y_test, y_pred, y_pred_prob)

## Zusammenfassung

In diesem Notebook haben wir Transfer Learning mit einem vortrainierten CNN (MobileNetV2) implementiert, um Autos im CIFAR-10 Datensatz zu erkennen. Hier sind die wichtigsten Schritte und Ergebnisse:

1. **Vorbereitung der Daten**: Wir haben die vorbereiteten Daten aus dem ersten Notebook geladen und sie für das vortrainierte Modell vorbereitet, indem wir sie auf 224x224 Pixel skaliert und die entsprechende Vorverarbeitung angewendet haben.

2. **Transfer Learning**: Wir haben das vortrainierte MobileNetV2-Modell geladen, seine Schichten eingefroren und unsere eigenen benutzerdefinierten Schichten für die binäre Klassifikation hinzugefügt.

3. **Training**: Wir haben das angepasste Modell mit einer niedrigen Lernrate trainiert, um die vortrainierten Merkmale nicht zu stören, und Early Stopping verwendet, um Overfitting zu vermeiden.

4. **Evaluierung**: Wir haben das Modell auf den Testdaten evaluiert und eine gute Genauigkeit erreicht, was die Effektivität des Transfer Learning für diese Aufgabe zeigt.

5. **Visualisierung**: Wir haben den Trainingsverlauf und einige Vorhersagen visualisiert, um ein besseres Verständnis für die Leistung des Modells zu bekommen.

Transfer Learning hat sich als effektive Methode erwiesen, um ein leistungsfähiges Modell für die Autoerkennung zu erstellen, ohne ein komplexes Modell von Grund auf trainieren zu müssen. Dies ist besonders nützlich, wenn begrenzte Daten oder Rechenressourcen verfügbar sind.

Im nächsten Notebook werden wir unser trainiertes Modell verwenden, um Autos auf größeren, realistischeren Bildern zu erkennen.